In [ ]:
# %%
import sys
sys.path.append("../src")

In [ ]:
# %% 
import time, os, json
from pathlib import Path
import pandas as pd
import numpy as np
import itertools
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib
from sklearn.model_selection import ParameterSampler

from FIT_python.pipeline_individual_id.pairwise_individual_id_pipeline import (
    run_all_pairwise_projections_parallel
)
from FIT_python.pipeline_individual_id.generate_trails_and_trailpairs import (
    generate_pairwise_comparisons_from_df
)

# %% 1) Basis-Pfade & Hyperparameter-Räume
splits_root = Path("data/splits/eurasian_otter")
models_root = Path(r"results/data/random_search_standard_metrics/best_maj_test_pct")
output_root = Path("results/data/pairwise_comp_eurasian_otter")
output_root.mkdir(parents=True, exist_ok=True)

feature_counts    = [2,5,10,15,50,20,50,100,200]
selection_methods = ["random_forest","forward","univariate","variance","lasso"]
reducers          = ["pca","lda","umap"]
n_components_list = [2, 3, 5, 10]
outlier_methods   = [None,"clip","zscore"]
scaler_methods    = [None,"standard","robust"]
use_sex_opts      = [False, True]
n_jobs            = -1

# %% 2) Grid erzeugen & 100 auswählen
grid = pd.DataFrame(
    itertools.product(
        feature_counts,
        selection_methods,
        reducers,
        n_components_list,
        outlier_methods,
        scaler_methods,
        use_sex_opts
    ),
    columns=[
        "feature_count",
        "selection_method",
        "reducer",
        "n_components",
        "outlier_method",
        "scaler_method",
        "use_sex"
    ]
).sort_values("feature_count", ascending=False).reset_index(drop=True)

sampled_grid = grid.sample(n=2, random_state=42)
sampled_grid = sampled_grid.sort_values("feature_count", ascending=False).reset_index(drop=True)

# === Zusätzliche Standard-Methoden einfügen ===
std_feature_counts   = [10, 12, 14, 16, 18, 20]
std_selection_method = "forward"
std_reducer          = "lda"
std_n_components     = 2     # bei LDA nicht relevant
std_outlier_method   = None
std_scaler_method    = None

extra = []
for fc in std_feature_counts:
    for use_sex in [False, True]:
        extra.append({
            "feature_count":    fc,
            "selection_method": std_selection_method,
            "reducer":          std_reducer,
            "n_components":     std_n_components,
            "outlier_method":   std_outlier_method,
            "scaler_method":    std_scaler_method,
            "use_sex":          use_sex
        })

extra_df = pd.DataFrame(extra)

# an sampled_grid anhängen und wieder sortieren
sampled_grid = pd.concat([sampled_grid, extra_df], ignore_index=True)
sampled_grid = sampled_grid.sort_values("feature_count", ascending=False).reset_index(drop=True)

# %% 3) Spezies-Schleife
species_dir = Path(splits_root)
    species = 'eurasian_otter'
    print(f"\n🔄 Processing species: {species}")

    # 3.1) Daten laden
    df = pd.read_csv(species_dir / "train.csv")

    # 3.2) Feature-Spalten wählen
    if species == "eurasian_otter_(1)":
        feature_cols = [
            c for c in df.columns 
            if c.startswith(("dist","ang","t")) and pd.api.types.is_numeric_dtype(df[c])
        ]
    else:
        feature_cols = [
            c for c in df.columns[5:]
            if pd.api.types.is_numeric_dtype(df[c])
        ]
    print(f"  → {len(feature_cols)} numerische Features")

    # 3.3) Pairs & Summary generieren
    comps, summary = generate_pairwise_comparisons_from_df(
        df,
        id_col="individual_id",
        chunk_size=10,
        trail_size_list=[9,7,5,3,2],
        max_individuals=3,
        max_trails_per_animal=2,
        n_folds=3,
        random_state=42
        fallback_col='trail',
        sampling_mode='window',
    )
    print("  Summary der Trail-Längen:")
    display(summary)

    # 3.4) Resume-Logik für species-spezifisches Parquet
    results_fp = output_root / f"{species}_all_results_combined.parquet"
    if results_fp.exists():
        done = pd.read_parquet(results_fp, columns=sampled_grid.columns)
        done_keys = set(tuple(r) for r in done.values)
        todo = sampled_grid[
            sampled_grid.apply(lambda row: tuple(row.values) not in done_keys, axis=1)
        ].reset_index(drop=True)
    else:
        todo = sampled_grid.copy()

    print(f"  → Noch offen: {len(todo)}/{len(sampled_grid)} Runs")

    # 3.5) Hyper-Parameter-Loop
    for idx, row in todo.iterrows():
        fc, sel, red, nc, out, scl, sex = (
            int(row.feature_count),
            row.selection_method,
            row.reducer,
            int(row.n_components),
            row.outlier_method,
            row.scaler_method,
            bool(row.use_sex)
        )
        desc = (
            f"FC={fc}|FS={sel}|Red={red}/{nc}|Out={out}|Scl={scl}|Sex={'on' if sex else 'off'}"
        )
        print(f"    → Run {idx+1}/{len(todo)}: {desc}")
        t0 = time.time()

        # gewrappt in Listen:
        out_list = [out] if out is not None else [None]
        scl_list = [scl] if scl is not None else [None]

        with tqdm_joblib(tqdm(desc=desc, total=len(comps), leave=False)):
            res = run_all_pairwise_projections_parallel(
                comparisons=comps,
                df=df,
                feature_cols=feature_cols,
                k_features=fc,
                selection_method=sel,
                reducers=[red],           # bleibt wie gehabt
                n_components=[nc],        # bleibt wie gehabt
                outlier_methods=out_list, # **jetzt eine Liste**
                scaler_methods=scl_list,  # **jetzt eine Liste**
                use_sexmodel_prediction=sex,
                sexmodel_path=(
                    models_root / f"{species}.joblib"
                    if sex else None
                ),
                debug=False,
                n_jobs=n_jobs
            )
        print(f"      → {len(res)} Rows in {time.time()-t0:.1f}s")

        # 3.6) Results anreichern & speichern
        df_run = pd.DataFrame(res)
        for col in sampled_grid.columns:
            df_run[col] = row[col]
        if results_fp.exists():
            df_full = pd.concat([pd.read_parquet(results_fp), df_run], ignore_index=True)
        else:
            df_full = df_run
        df_full.to_parquet(results_fp, index=False)

    print(f"  ✅ Fertig für {species}, Parquet: {results_fp}")

# %% 4) Ergebnisse
all_parquets = [results_fp]
df_all = pd.concat((pd.read_parquet(p) for p in all_parquets), ignore_index=True)
df_all.to_parquet(output_root / 'eurasian_otter_all_results_combined.parquet', index=False)
print("\n🎉 Fertig – Ergebnis-Parquet erzeugt:")
print(f"   {output_root/'eurasian_otter_all_results_combined.parquet'}")
display(df_all.head())

In [ ]:
# %% 
import time, os, json
from pathlib import Path
import pandas as pd
import numpy as np
import itertools
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib
from sklearn.model_selection import ParameterSampler


from FIT_python.pipeline_individual_id.pairwise_individual_id_pipeline import (
    run_all_pairwise_projections_parallel
)
from FIT_python.pipeline_individual_id.generate_trails_and_trailpairs import (
    generate_pairwise_comparisons_from_df
)

# %% 1) Basis-Pfade & Hyperparameter-Räume
splits_root = Path("data/splits/eurasian_otter")
models_root = Path(r"results/data/sex_models_best")
output_root = Path("results/data/individual_id/eurasian_otter_baseline_fit")
output_root.mkdir(parents=True, exist_ok=True)



# BAseline FIT Method
baseline_feature_counts = [10, 12, 14, 16, 18, 20]

# Erzeuge alle Kombinationen für use_sex = False und True
baseline_models = pd.DataFrame(
    list(itertools.product(
        baseline_feature_counts,         # feature_count
        ["forward"],                     # selection_method
        ["lda"],                         # reducer
        [2],                             # n_components
        [None],                          # outlier_method
        [None],                          # scaler_method
        [False, True]                    # use_sex
    )),
    columns=[
        "feature_count",
        "selection_method",
        "reducer",
        "n_components",
        "outlier_method",
        "scaler_method",
        "use_sex"
    ]
)



feature_counts    = [2,5,8,15,50,100,200]
selection_methods = ["random_forest","forward","univariate","variance","lasso"]
reducers          = ["pca","lda","umap"]
n_components_list = [2, 3, 5, 10]
outlier_methods   = [None,"clip","zscore"]
scaler_methods    = [None,"standard","robust"]
use_sex_opts      = [False, True]
n_jobs            = 6

# %% 2) Grid erzeugen & 100 auswählen via ParameterSampler
from sklearn.model_selection import ParameterSampler

param_dist = {
    "feature_count":    feature_counts,
    "selection_method": selection_methods,
    "reducer":          reducers,
    "n_components":     n_components_list,
    "outlier_method":   outlier_methods,
    "scaler_method":    scaler_methods,
    "use_sex":          use_sex_opts,
}

# Ziehe 138 zufällige Konfigurationen
sampled_grid = pd.DataFrame(
    list(ParameterSampler(param_dist, n_iter=20, random_state=42))
)

# Sortieren wie gehabt
sampled_grid = sampled_grid.sort_values("feature_count", ascending=False).reset_index(drop=True)

# ➕ Anhängen an das bestehende Grid
sampled_grid_extended = pd.concat([sampled_grid, baseline_models], ignore_index=True)

# 📊 Optional sortieren
sampled_grid_extended = sampled_grid_extended.sort_values("feature_count", ascending=False).reset_index(drop=True)

# %% 3) Ergebnis-Datei initialisieren oder laden
out_fp = output_root / "eurasian_otter_all_results_combined.parquet"
if out_fp.exists():
    df_completed = pd.read_parquet(out_fp)
else:
    df_completed = pd.DataFrame()  # wird später mit den ersten Runs gefüllt

# %% 4) Spezies-Schleife
species_dir = Path(splits_root)
    species = 'eurasian_otter'
    print(f"\n🔄 Processing species: {species}")

    # 4.1) Daten laden
    df = pd.read_csv(species_dir / "train.csv")

    # 4.2) Feature-Spalten wählen
    if species == "eurasian_otter_(1)":
        feature_cols = [
            c for c in df.columns 
            if c.startswith(("dist","ang","t")) and pd.api.types.is_numeric_dtype(df[c])
        ]
    else:
        feature_cols = [
            c for c in df.columns[5:]
            if pd.api.types.is_numeric_dtype(df[c])
        ]
    print(f"  → {len(feature_cols)} numerische Features")

    # 4.3) Pairs & Summary generieren (einmal pro Spezies)
    comps, summary = generate_pairwise_comparisons_from_df(
        df,
        id_col="individual_id",
        chunk_size=10,
        trail_size_list=[9,7,5,3,2],
        max_individuals=3,
        max_trails_per_animal=3,
        n_folds=3,
        random_state=42
        fallback_col='trail',
        sampling_mode='window',
    )
    print("  Summary der Trail-Längen:")
    display(summary)

    # 4.4) Hyper-Parameter-Loop
    for idx, row in sampled_grid_extended.iterrows():
        fc, sel, red, nc, out, scl, sex = (
            int(row.feature_count),
            row.selection_method,
            row.reducer,
            int(row.n_components),
            row.outlier_method,
            row.scaler_method,
            bool(row.use_sex)
        )
        # Key-Tuple, um zu prüfen, ob wir das schon berechnet haben
        run_key = (
            species, fc, sel, red, nc, out, scl, sex
        )
        already = False
        if not df_completed.empty:
            # existiert mindestens eine Zeile mit genau diesen Parametern?
            mask = (
                (df_completed["species"] == species) &
                (df_completed["feature_count"] == fc) &
                (df_completed["selection_method"] == sel) &
                (df_completed["reducer"] == red) &
                (df_completed["n_components"] == nc) &
                (df_completed["outlier_method"] == out) &
                (df_completed["scaler_method"] == scl) &
                (df_completed["use_sex"] == sex)
            )
            if mask.any():
                already = True

        if already:
            print(f"    → SKIP {idx+1}/{len(sampled_grid)}  (bereits vorhanden)")
            continue

        desc = (
            f"FC={fc}|FS={sel}|Red={red}/{nc}|Out={out}|Scl={scl}|Sex={'on' if sex else 'off'}"
        )
        print(f"    → Run {idx+1}/{len(sampled_grid)}: {desc}")
        t0 = time.time()

        with tqdm_joblib(tqdm(desc=desc, total=len(comps), leave=False)):
            res = run_all_pairwise_projections_parallel(
                comparisons=comps,
                df=df,
                feature_cols=feature_cols,
                k_features=fc,
                selection_method=sel,
                reducers=[red],
                n_components=[nc],
                outlier_methods=out,
                scaler_methods=scl,
                use_sexmodel_prediction=sex,
                sexmodel_path=(
                    models_root / f"{species}.joblib"
                    if sex else None
                ),
                debug=False,
                n_jobs=n_jobs
            )
        dur = time.time() - t0
        print(f"      → {len(res)} Rows in {dur:.1f}s")

        # 4.5) Results anreichern und anhängen
        df_run = pd.DataFrame(res)
        df_run["species"]          = species
        df_run["duration_s"]       = dur
        df_run["feature_count"]    = fc
        df_run["selection_method"] = sel
        df_run["reducer"]          = red
        df_run["n_components"]     = nc
        df_run["outlier_method"]   = out
        df_run["scaler_method"]    = scl
        df_run["use_sex"]          = sex

        # an df_completed in memory anhängen
        df_completed = pd.concat([df_completed, df_run], ignore_index=True)
        # sofort ins Parquet schreiben
        df_completed.to_parquet(out_fp, index=False)

# %% 5) Abschließende Kontrolle
print(f"\n🎉 Fertig – Gesamt-Parquet (inkrementell) aktualisiert unter:\n {out_fp}")
display(df_completed.head())